In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from air_quality_monitor.analysis import AirQualityAnalyser
from air_quality_monitor.storage import CSVStorage
from air_quality_monitor.models import AirQualityReading
from pathlib import Path
import pandas as pd
from datetime import datetime
#import csv

In [ ]:
notebook_dir = Path().resolve()
filepath = notebook_dir.parent / "data" / "aqi_history.csv"

storage = CSVStorage(filepath, AirQualityReading)

df = storage.read()
df['hour'] = df['pollutant_timestamp'].dt.hour
df['dow'] = df['pollutant_timestamp'].dt.dayofweek
print(df)
print(df.dtypes)

In [ ]:
# Plot line charts showing AQI over time for different cities
analyser = AirQualityAnalyser(df)
df_to_print = pd.DataFrame()
for city in ["Berlin", "Athens", "Stockholm"]:
    df_city = analyser.filter_by_city(city).get_aqi_by_date_range(start_date=datetime(2026, 3, 24, 9, 0))
    df_city['city'] = city
    print(df_city.shape)
    print(df_city.columns)
    df_to_print = pd.concat([df_to_print, df_city])

print(df_to_print)
fig = analyser.plot_line_chart(df_to_print, x_axis="pollutant_timestamp", y_axis="aqi", title="AQI over time")
fig.show()

In [ ]:
df_sj = df[df['city'] == 'Sarajevo']
df_sj = df_sj[['city', 'hour', 'aqi']]
#print(df_sj)
#print(df_sj.shape)
df_sj_grouped = df_sj.groupby(['city', 'hour']).mean().reset_index()
df_sj_grouped.rename(columns={'aqi': 'mean_aqi'}, inplace=True)
#print(df_sj_grouped)
#print(df_sj_grouped.shape)

fig = analyser.plot_line_chart(df_sj_grouped, x_axis='hour', y_axis='mean_aqi', title="Mean AQI over time")
fig.show()

In [ ]:
df_aqi = df[['city', 'hour', 'aqi']]
df_aqi_grouped = df_aqi.groupby(['city', 'hour']).mean().reset_index()
df_aqi_grouped.rename(columns={'aqi': 'mean_aqi'}, inplace=True)

fig = analyser.plot_line_chart(df_aqi_grouped, x_axis='hour', y_axis='mean_aqi', title="Mean AQI over time")
fig.show()